# NISQA-SIM Low-MOS Mix Generator

This notebook creates **40 mixed files** from `NISQA_TRAIN_SIM`, using only low-MOS source pairs
so degradations are more obvious.

Key behavior:
- Mix-only output (no REF-only / DEG-only files)
- Low-MOS sampling (`mos <= MOS_MAX_THRESHOLD`)
- Natural durations from source audio (no fixed-length padding)
- Randomized 1-3 degraded intervals per file
- Table + waveform visualization of generated mixes


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import math
import random

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import resample_poly

In [ ]:
@dataclass
class Segment:
    """Represents a degradation interval in seconds."""

    start: float
    end: float


DATA_ROOT = Path("../data/raw/NISQA_Corpus")
SIM_SPLIT = "NISQA_TRAIN_SIM"
CSV_PATH = DATA_ROOT / SIM_SPLIT / f"{SIM_SPLIT}_file.csv"
OUTPUT_DIR = Path("../data/processed/nisqa_sim_mix_lowmos_40")
MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"

TOTAL_MIX_FILES = 40
MOS_MAX_THRESHOLD = 2.2
REQUIRE_ACTIVE_DEGRADATION_TYPES = True

TARGET_SAMPLE_RATE = 16000
MIN_DEG_SEGMENTS = 1
MAX_DEG_SEGMENTS = 3
MAX_DURATION_SECONDS = None
SEED = 42
PLOT_EXAMPLES = TOTAL_MIX_FILES

DEGRADATION_COLUMNS = [
    "filter",
    "timeclipping",
    "wbgn",
    "p50mnru",
    "bgn",
    "clipping",
    "arb_filter",
    "codec1",
    "codec2",
    "codec3",
    "plcMode1",
    "plcMode2",
    "plcMode3",
]

rng = random.Random(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_audio_mono(path: Path) -> tuple[np.ndarray, int]:
    """Load waveform and convert to mono float32."""

    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    return audio.astype(np.float32), int(sr)


def resample_if_needed(audio: np.ndarray, sr_in: int, sr_out: int) -> np.ndarray:
    """Resample when the input sample rate differs from target rate."""

    if sr_in == sr_out:
        return audio
    gcd = math.gcd(sr_in, sr_out)
    up = sr_out // gcd
    down = sr_in // gcd
    return resample_poly(audio, up=up, down=down).astype(np.float32)


def align_pair(ref_audio: np.ndarray, deg_audio: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Align REF and DEG waveforms to a shared length without padding."""

    length = min(len(ref_audio), len(deg_audio))
    if MAX_DURATION_SECONDS is not None:
        max_len = int(round(MAX_DURATION_SECONDS * TARGET_SAMPLE_RATE))
        length = min(length, max_len)

    ref_audio = ref_audio[:length]
    deg_audio = deg_audio[:length]
    return ref_audio, deg_audio


def random_segments(total_seconds: float) -> list[Segment]:
    """Generate non-overlapping random degradation intervals."""

    if total_seconds <= 0.3:
        return [Segment(0.0, total_seconds)]

    max_possible = max(1, int(total_seconds // 0.35))
    n_segments = min(rng.randint(MIN_DEG_SEGMENTS, MAX_DEG_SEGMENTS), max_possible)

    min_len = max(0.18, total_seconds * 0.06)
    max_len = max(min_len, total_seconds * 0.55)

    accepted: list[Segment] = []
    attempts = 0
    while len(accepted) < n_segments and attempts < 400:
        attempts += 1
        seg_len = rng.uniform(min_len, max_len)
        seg_len = min(seg_len, total_seconds)
        start = rng.uniform(0.0, max(0.0, total_seconds - seg_len))
        candidate = Segment(start=start, end=start + seg_len)

        overlaps = any(
            not (candidate.end <= seg.start or candidate.start >= seg.end)
            for seg in accepted
        )
        if overlaps:
            continue

        accepted.append(candidate)

    if not accepted:
        fallback_len = min(total_seconds, max(0.2, total_seconds * 0.4))
        accepted = [Segment(0.0, fallback_len)]

    return sorted(accepted, key=lambda s: s.start)


def build_mix(ref_audio: np.ndarray, deg_audio: np.ndarray, sr: int) -> tuple[np.ndarray, list[Segment]]:
    """Create a mixed waveform by inserting DEG intervals into REF audio."""

    mixed = ref_audio.copy()
    total_seconds = len(mixed) / sr
    segments = random_segments(total_seconds=total_seconds)

    for seg in segments:
        i0 = max(0, min(int(round(seg.start * sr)), len(mixed)))
        i1 = max(i0, min(int(round(seg.end * sr)), len(mixed)))
        mixed[i0:i1] = deg_audio[i0:i1]

    return mixed, segments


def serialize_segments(segments: list[Segment]) -> str:
    """Serialize intervals to JSON for manifest storage."""

    payload = [{"start": round(s.start, 3), "end": round(s.end, 3)} for s in segments]
    return json.dumps(payload)


def timeline_from_deg_segments(total_seconds: float, deg_segments: list[Segment]) -> list[dict]:
    """Build an ordered REF/DEG timeline from degradation segments."""

    timeline: list[dict] = []
    cursor = 0.0

    for seg in sorted(deg_segments, key=lambda s: s.start):
        start = max(0.0, min(total_seconds, float(seg.start)))
        end = max(start, min(total_seconds, float(seg.end)))

        if start > cursor:
            timeline.append({"start": round(cursor, 3), "end": round(start, 3), "source": "REF"})

        if end > start:
            timeline.append({"start": round(start, 3), "end": round(end, 3), "source": "DEG"})
            cursor = end

    if cursor < total_seconds:
        timeline.append({"start": round(cursor, 3), "end": round(total_seconds, 3), "source": "REF"})

    return timeline


def switch_points_from_timeline(timeline: list[dict]) -> list[float]:
    """Return timeline boundaries where source may switch."""

    points: list[float] = []
    for item in timeline:
        points.extend([float(item["start"]), float(item["end"])])
    return sorted(set(round(p, 3) for p in points))


def is_active_tag(value: object) -> bool:
    """Return True when a metadata field indicates an active degradation."""

    if pd.isna(value):
        return False
    token = str(value).strip()
    return token not in {"", "-", "nan", "None"}


def extract_active_degradations(row: pd.Series) -> list[str]:
    """Extract active degradation tags from NISQA SIM metadata columns."""

    return [col for col in DEGRADATION_COLUMNS if is_active_tag(row.get(col, np.nan))]


In [ ]:
df = pd.read_csv(CSV_PATH)
df["ref_path"] = df["filepath_ref"].apply(lambda p: DATA_ROOT / p)
df["deg_path"] = df["filepath_deg"].apply(lambda p: DATA_ROOT / p)

df = df[df["ref_path"].apply(Path.exists) & df["deg_path"].apply(Path.exists)].copy()
df = df[df["mos"].notna() & (df["mos"] <= MOS_MAX_THRESHOLD)].copy()
df = df.reset_index(drop=True)

df["active_degradation_types"] = df.apply(extract_active_degradations, axis=1)
df["num_source_degradation_types"] = df["active_degradation_types"].apply(len)

if REQUIRE_ACTIVE_DEGRADATION_TYPES:
    df = df[df["num_source_degradation_types"] > 0].copy().reset_index(drop=True)

if len(df) < TOTAL_MIX_FILES:
    raise ValueError(
        f"Need at least {TOTAL_MIX_FILES} low-MOS rows after filtering, found {len(df)}. "
        "Try increasing MOS_MAX_THRESHOLD."
    )

chosen_idx = rng.sample(list(df.index), TOTAL_MIX_FILES)
selected = df.loc[chosen_idx].reset_index(drop=True)
selected["active_degradation_types_json"] = selected["active_degradation_types"].apply(json.dumps)

print(f"Selected {len(selected)} source pairs from {SIM_SPLIT} with MOS <= {MOS_MAX_THRESHOLD}.")
print(f"Selected MOS range: {selected['mos'].min():.2f} to {selected['mos'].max():.2f}")

active_tag_counts = selected["active_degradation_types"].explode().value_counts()
if len(active_tag_counts) > 0:
    display(active_tag_counts.rename("count").to_frame())

display(
    selected[[
        "filename_deg",
        "mos",
        "active_degradation_types",
        "num_source_degradation_types",
    ]].sort_values("mos").reset_index(drop=True)
)


In [ ]:
records: list[dict] = []

for idx, row in selected.iterrows():
    ref_audio, ref_sr = load_audio_mono(row["ref_path"])
    deg_audio, deg_sr = load_audio_mono(row["deg_path"])

    ref_audio = resample_if_needed(ref_audio, ref_sr, TARGET_SAMPLE_RATE)
    deg_audio = resample_if_needed(deg_audio, deg_sr, TARGET_SAMPLE_RATE)
    ref_audio, deg_audio = align_pair(ref_audio, deg_audio)

    mixed_audio, deg_segments = build_mix(ref_audio, deg_audio, TARGET_SAMPLE_RATE)

    stem = Path(row["filename_deg"]).stem
    out_path = OUTPUT_DIR / f"{idx:03d}_mix_{stem}.wav"
    sf.write(out_path, mixed_audio, TARGET_SAMPLE_RATE)

    duration_seconds = round(len(mixed_audio) / TARGET_SAMPLE_RATE, 3)
    text_segments = json.dumps([{"start": 0.0, "end": duration_seconds}])
    timeline = timeline_from_deg_segments(duration_seconds, deg_segments)
    switch_points = switch_points_from_timeline(timeline)

    records.append({
        "index": idx,
        "filename_ref": row["filename_ref"],
        "filename_deg": row["filename_deg"],
        "mos": row["mos"],
        "duration_seconds": duration_seconds,
        "text_segments": text_segments,
        "mix_deg_segments": serialize_segments(deg_segments),
        "switch_points": json.dumps(switch_points),
        "mix_timeline": json.dumps(timeline),
        "source_degradation_types": row["active_degradation_types_json"],
        "num_source_degradation_types": int(row["num_source_degradation_types"]),
    })

manifest_df = pd.DataFrame(records).sort_values("index").reset_index(drop=True)
manifest_df.to_csv(MANIFEST_PATH, index=False)

print(f"Wrote {len(manifest_df)} mixed files to {OUTPUT_DIR}")
print(f"Manifest: {MANIFEST_PATH}")

display(manifest_df[["mos", "duration_seconds", "num_source_degradation_types"]].describe())
display(
    manifest_df[[
        "index",
        "filename_deg",
        "mos",
        "mix_deg_segments",
        "source_degradation_types",
    ]]
)


In [ ]:
table_df = manifest_df[[
    "index",
    "filename_deg",
    "mos",
    "mix_deg_segments",
    "source_degradation_types",
]].sort_values("index").reset_index(drop=True)

display(table_df)

plot_df = manifest_df.sort_values("index").head(min(PLOT_EXAMPLES, len(manifest_df)))

for _, row in plot_df.iterrows():
    print()
    stem = Path(row["filename_deg"]).stem
    mix_path = OUTPUT_DIR / f"{int(row['index']):03d}_mix_{stem}.wav"

    mix_audio, mix_sr = load_audio_mono(mix_path)
    mix_audio = resample_if_needed(mix_audio, mix_sr, TARGET_SAMPLE_RATE)

    deg_segments = json.loads(row["mix_deg_segments"])
    active_types = json.loads(row["source_degradation_types"])

    times = np.arange(len(mix_audio)) / TARGET_SAMPLE_RATE
    fig, ax = plt.subplots(figsize=(12, 3.2))

    for seg in deg_segments:
        ax.axvspan(seg["start"], seg["end"], alpha=0.25, color="#ffb347")

    ax.plot(times, mix_audio, linewidth=0.65, color="#111111")
    ax.set_title("MIX waveform (orange background = DEG region)")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")

    types_label = ", ".join(active_types) if active_types else "none"
    fig.suptitle(
        f"Example {int(row['index']):03d} | MOS={row['mos']:.2f} | active={types_label}",
        y=1.04,
    )
    plt.tight_layout()
    plt.show()

    print(f"mix_deg_segments={row['mix_deg_segments']}")
    display(Audio(filename=str(mix_path)))


## Output

- Audio files: `../data/processed/nisqa_sim_mix_lowmos_40/*.wav`
- Manifest: `../data/processed/nisqa_sim_mix_lowmos_40/manifest.csv`

Manifest schema:
- `index`
- `filename_ref`
- `filename_deg`
- `mos`
- `duration_seconds`
- `text_segments`
- `mix_deg_segments`
- `switch_points`
- `mix_timeline`
- `source_degradation_types`
- `num_source_degradation_types`

Low-MOS source filter:
- `mos <= MOS_MAX_THRESHOLD` (default: `2.2`)
- `num_source_degradation_types > 0` when `REQUIRE_ACTIVE_DEGRADATION_TYPES=True`
